# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [11]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [12]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [13]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [14]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'company/partner page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [15]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [16]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [17]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discord community', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'Community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [18]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [19]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.8-27B
Updated
12 days ago
•
2.95M
•
12.8k
unsloth/Qwen3.8-27B-GGUF
Updated
6 days ago
•
7.33M
•
2.94k
OBLITERATUS/Qwen3.8-27B-OBLITERATED
Updated
1 day ago
•
390k
•
770
orcarouter/Qwen3.8-27B-Uncensored-MLX
Updated
2 days ago
•
68.9k
•
1.12k
orcarouter/Qwen3.8-27B-Uncensore

In [27]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [21]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.8-27B\nUpdated\n12 days ago\n•\n2.95M\n•\n12.8k\nunsloth/Qwen3.8-27B-GGUF\nUpdated\n6 days ago\n•\n7.33M\

In [28]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the thriving collaboration platform at the forefront of the AI revolution, building the future of machine learning. It serves as a vibrant community hub where machine learning engineers, scientists, researchers, and developers come together to create, share, and advance open-source models, datasets, and applications.

The company’s core mission is to empower the next generation of AI innovators by providing a central space that fosters collaboration, innovation, and ethical AI development. Hugging Face stands as a leading force in open and responsible artificial intelligence, with tools and libraries that are widely adopted across the global AI landscape.

---

## What We Offer

### Platform Features
- **Models:** Access and contribute to over 2 million machine learning models spanning diverse AI tasks and use cases.
- **Datasets:** Browse and share more than 500,000 curated datasets supporting wide-ranging AI research and applications.
- **Spaces:** Launch and discover AI-powered applications built by the community to explore innovative ML solutions.
- **Buckets:** Secure and scalable storage solutions tailored for machine learning workflows.
- **HuggingChat:** An interactive conversational AI platform part of Hugging Face’s expanding ecosystem.
- **Inference Endpoints & Providers:** Production-ready services enabling enterprises to deploy and scale AI models efficiently.
- **Enterprise Solutions:** Tailored support and advanced offerings via Hugging Face PRO and enterprise partnerships.

### Community & Collaboration
- The platform hosts a **fast-growing, active AI community** engaged through Discord, forums, and GitHub.
- Continuous knowledge sharing through **blogs, daily research papers, educational content**, and open discussions.
- Encourages ethical AI development and open science, making it possible for anyone—from beginners to experts—to contribute safely.

---

## Our Culture

- **Open-Source First:** At the heart of Hugging Face is a strong commitment to openness and transparency in AI.
- **Ethical AI:** Dedication to building AI tools that prioritize fairness, inclusivity, and responsibility.
- **Collaboration & Education:** A learning-driven culture that nurtures growth through community interaction and shared expertise.
- **Diversity & Inclusion:** Embracing varied perspectives to enhance creativity and technical innovation.
- **Impact-Driven:** Focused on pushing the boundaries of AI to solve real-world problems and accelerate progress in the field.

---

## Who Uses Hugging Face?

- **Machine Learning Engineers & Researchers:** Leverage the repository of models and datasets to accelerate development and experimentation.
- **Data Scientists & AI Practitioners:** Build custom AI applications using community-driven tools and resources.
- **Enterprises & Startups:** Integrate scalable AI services for production use through enterprise-grade infrastructure.
- **Educators & Students:** Access rich educational resources to learn and teach machine learning.
- **AI Enthusiasts & Developers:** Participate actively in the community through collaborative projects, challenges, and events.

---

## Careers & Opportunities

Hugging Face is constantly growing and looking for passionate individuals who want to shape the future of AI:

- Positions in Research, Engineering, Product, and Community Engagement.
- Collaborative and inclusive work environment embracing remote and hybrid work models.
- Work with cutting-edge AI technologies alongside one of the most innovative teams in the industry.
- Opportunity to contribute to open source and impact millions of users globally.
- Competitive benefits and professional growth pathways.

Explore current openings and join us:
**[huggingface.co/careers](https://huggingface.co/careers)**

---

## Join the Revolution

Whether you are an AI researcher, developer, enterprise client, or simply passionate about machine learning, Hugging Face welcomes you to:

- Discover and deploy state-of-the-art ML models.
- Collaborate on datasets and AI applications.
- Learn and grow within a diverse and ethical AI community.
- Build the future of machine learning together.

**Website:** https://huggingface.co  
**Community:** Discord, Forum, GitHub, Twitter, LinkedIn

---

*Hugging Face - The AI community building the future.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [29]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a leading AI company and the premier collaboration platform for the machine learning community. Dedicated to building the future of artificial intelligence, Hugging Face empowers ML engineers, scientists, and users worldwide to share, explore, and innovate together.

At its core is the Hugging Face Hub — the central platform where millions of open-source models, datasets, and applications are shared and developed collaboratively. Emphasizing openness, accessibility, and ethics, Hugging Face is a pivotal force in the AI revolution, continuously pushing the boundaries at the cutting edge of AI technology.

---

## What We Offer

- **2M+ Open-Source Models:** A vast repository of machine learning models in various modalities such as text, image, audio, and multimodal applications.
- **500K+ Datasets:** Extensive curated datasets to train, validate, and benchmark models.
- **Spaces:** Deploy and demo ML apps directly in the browser via an easy-to-use interface tied to the Hub.
- **HuggingChat:** An interactive chat interface for AI applications.
- **Inference Endpoints & Providers:** Scalable serving infrastructure for model deployment tailored to enterprise needs.
- **Enterprise & Team Solutions:** Collaboration and infrastructure plans tailored for businesses seeking to leverage AI at scale.
- **Extensive Developer Resources:** Documentation, community forums, blogs, and educational material empowering users to learn and build confidently.

---

## Company Culture

Hugging Face thrives on a dynamic community spirit grounded in:

- **Open Source Ethos:** Encouraging sharing and transparency to foster innovation.
- **Collaborative Innovation:** Providing tools and spaces that nurture team-based and community contributions.
- **Ethical AI:** Committed to creating responsible, fair, and safe AI applications.
- **Continuous Learning:** Supporting the next generation of ML practitioners with accessible resources and community support.
- **Diverse & Inclusive:** A welcoming environment where diverse skills and perspectives drive excellence.

---

## Customers & Community

From startups, researchers, academia to Fortune 500 enterprises, Hugging Face serves a broad spectrum of users who depend on its platform to:

- Accelerate AI research and development.
- Build custom AI-powered applications.
- Deploy production-ready machine learning at scale.
- Foster community-driven innovation through open science.

The Hugging Face community includes millions of developers and organizations worldwide collaborating in an ever-expanding ecosystem of models, datasets, and AI-powered solutions.

---

## Career Opportunities

Hugging Face is constantly seeking passionate professionals who want to shape the future of AI. Joining Hugging Face means:

- Working at the intersection of cutting-edge research and real-world impact.
- Collaborating with talented scientists, engineers, and community leaders.
- Growing in a culture that values creativity, learning, and inclusion.
- Contributing to ethical and open AI technologies used by millions.

---

## Connect With Us

- **Website:** https://huggingface.co  
- **Community:** Active on Discord, GitHub, Twitter, and LinkedIn  
- **Learn & Engage:** Access tutorials, forums, documentation, and daily AI papers through the platform  
- **Enterprise Contact:** Custom plans and support available for businesses adopting AI  

---

**Hugging Face** — Building the future of AI, together.  
Join us on the journey to open, ethical, and revolutionary machine learning.

In [30]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Welcome to Hugging Face  
*The AI community building the future one model at a time*  

---

## Who Are We?  
Imagine a bustling online playground where machine learning wizards, data sorcerers, and AI adventurers join forces. That’s Hugging Face — the friendliest neighborhood for over 2 million AI models, half a million datasets, and a million+ apps. We make ML collaboration as easy as sharing memes (but way smarter).  

Our platform is where AI minds meet to create, share, and discover the future of machine learning. Whether you want to chat with a bot, generate videos from text, or compose music just by typing lyrics—we've got the space and tools to make it happen.  

---

## What’s in Our Hugging Face?

- **Models, Models & More Models!**  
  Browse 2 million+ pre-trained models including the popular Qwen 3.8-27B series updated recently with millions of downloads and counting. From text generation to image editing—your AI playground is waiting.

- **Datasets Galore**  
  Dive into over 500,000 datasets, from ultra-fine web text collections to protein binder designs. With constantly fresh updates, research or build on rich, diverse data effortlessly.

- **Spaces & Apps**  
  Play with AI apps hosted in community-run Spaces—from converting images to videos to creating custom music studios. It’s like AI-themed amusement park rides, but without the long lines.

- **Enterprise & PRO Services**  
  Need AI-powered solutions at scale? Hugging Face PRO and enterprise support services provide robust infrastructure, seamless inference endpoints, and storage buckets so your business runs smoother than a freshly trained model.

---

## Culture: Where Code Meets Community & Smiles  
At Hugging Face, we believe AI should be as collaborative as a group chat with your smartest friends — without the endless “typing…” suspense. Our culture is as open as our datasets: inclusive, supportive, and buzzing with curiosity.  

Join our forums, hop on Discord, or contribute on GitHub. We’re playful but serious about pushing AI forward. Plus, our yellow and orange brand colors (#FFD21E & #FF9D00) reflect our sunny outlook and vibrant energy.  

---

## For Customers & Partners  
Whether you're a startup dreaming big or an enterprise aiming to innovate, Hugging Face tailors solutions to fit your AI ambitions. Access enterprise-grade support, streamlined inference providers, and custom storage — all backed by a passionate community building the future.  

---

## Careers: Help Us Hug the Future 🤗  
Do you speak fluent Python? Dream in neural networks? Want to be in a place where your code impacts millions globally? We’re always hunting for AI engineers, community managers, researchers, and dreamers who want to turn cutting-edge science into open, accessible tools.  

Apply to be part of a team where collaboration is king, every release feels like a party, and making AI more human is the daily mission.   

---

## Why Hugging Face?  
Because AI isn’t just about algorithms and numbers—it’s about people, passion, and a shared vision. We’re the home of machine learning, a hub where creativity meets code, and the place where you literally “hug” the future.  

**Explore today:**  
👉 [Browse 2M+ Models](https://huggingface.co/models)  
👉 [Join the Community](https://huggingface.co/join)  
👉 [Discover AI Apps](https://huggingface.co/spaces)  

*Hugging Face — where AI meets humanity.*  

---

*P.S. Join us and find out firsthand why our community hugs AI like an old friend — with open arms and endless enthusiasm.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>